# Predicting Next-Day Stock Direction Under Market Regime Shift

*A study in feature selection, distribution shift, and why simpler models sometimes win.*

---

## Introduction

Financial markets are among the hardest environments for machine learning. Prices are
noisy, largely efficient, and driven by countless interacting forces — earnings, macro
policy, sentiment, liquidity. The efficient-market view says next-day direction should be
close to unpredictable. And yet, even a *tiny* consistent edge over a coin flip compounds
into something valuable at scale. That tension is what makes this problem interesting.

This notebook tackles a binary classification task: given 27 engineered technical
indicators for a stock on a given day, predict whether its adjusted close will be **higher
(1)** or **lower (0)** the next trading day. The data covers 100 anonymized US equities.
No raw prices, tickers, or dates are provided — only computed signals — which keeps the
focus squarely on modeling rather than data mining the underlying names.

The single most important fact about this dataset shapes every decision that follows:

> **Training data spans 2000–2023. Test data spans 2024–2026.**

These are not the same market. The training window contains the dot-com collapse, the 2008
financial crisis, and the COVID crash. The test window is the post-pandemic recovery, a
rate-cut cycle, and the AI-driven rally. A model that simply memorizes what worked in
2000–2023 has no guarantee — and, as we'll see, little chance — of transferring cleanly to
2024–2026. This is a **distribution shift** problem as much as a prediction problem, and
treating it that way is the central theme of the analysis.


## Problem Understanding

A few properties of this task drive the modeling strategy. It's worth being explicit about
them up front, because they explain choices that would otherwise look arbitrary.

**The signal is weak by nature.** The competition framing is instructive: an AUC of 0.500
is random, and even 0.520 is described as *meaningful* in a real trading context. We are
not looking for a model that is "accurate" in any everyday sense. We are looking for a
model that ranks up-days above down-days *slightly* better than chance, and does so on data
it has never seen. When the signal is this faint, the dominant risk is not underfitting —
it's fitting noise and mistaking it for signal.

**Features overlap heavily.** Many of the 27 indicators are near-duplicates computed over
different windows. `return_5d` and `roc_5` measure the same 5-day momentum; the family of
`sma_ratio_*` features all express price relative to a moving average. Feeding a model
redundant columns doesn't add information — it inflates apparent importance and wastes
capacity. Some pruning is essential.

**The target is nearly balanced.** Roughly 50.3% up-days in training and 51.6% in test.
Balance is convenient (no resampling needed) but also a reminder of how little structural
edge exists to begin with.

**Evaluation is AUC-ROC.** We predict a *probability* per row and are scored on ranking
quality, not hard accuracy. This means calibration matters less than getting the relative
ordering right, and it makes probability-blending across models a natural tool.

Put together, these point away from "train the most powerful model possible" and toward
"build the most *robust* model possible." Those are different goals, and on data with a
regime shift they often conflict.


## Objective

The goal is a model that **generalizes across a market regime change** — one whose edge on
2024–2026 is real, not an artifact of patterns specific to 2000–2023.

Concretely, the analysis is organized around four questions:

1. **How different are the two periods, really?** Before modeling, quantify the distribution
   shift and identify which features carry the most era-specific information.
2. **Which features are both predictive and stable?** Rank features by genuine predictive
   contribution, then cross-reference against their shift risk. The features we want are
   the ones that predict well *and* look the same across time.
3. **Does removing shift-prone features actually help?** Test this directly rather than
   assuming it. Measure the trade-off between cross-validation score and held-out
   generalization.
4. **What model — or combination of models — is most robust?** Compare a single strong
   learner against ensembles, with particular attention to whether *diversity* between
   model families beats raw individual strength.

A recurring methodological point runs through all four: on regime-shifted data,
**cross-validation score is not a trustworthy proxy for real-world performance.** Much of
what follows is about noticing when the two diverge and choosing to trust the right one.


## Approach at a Glance

The pipeline proceeds in the following order. Each step feeds the next, and the reasoning
for each is spelled out where it appears.

1. **Configuration** — centralize every path and hyperparameter so the notebook is
   reproducible and easy to tune.
2. **Memory-optimized loading** — downcast types so the full pipeline fits comfortably in a
   16 GB kernel.
3. **Adversarial validation** — train a classifier to distinguish train from test rows,
   measuring distribution shift and flagging the features responsible.
4. **SHAP feature selection** — rank features by predictive contribution across three
   independent gradient-boosting libraries, keeping only what they agree on.
5. **Collinearity reduction** — drop redundant, near-duplicate features.
6. **Per-stock normalization** — z-score each signal within each stock so values mean the
   same thing across very different equities.
7. **Ablation study** — empirically test which feature configuration generalizes best.
8. **Ensemble modeling** — combine diverse model families and compare blending strategies.

Let's begin.


---

## 1. Environment Setup

Standard scientific-Python stack plus the three gradient-boosting libraries
(CatBoost, XGBoost, LightGBM) and SHAP for model-agnostic feature attribution.


In [ ]:
!pip install catboost shap xgboost lightgbm -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from matplotlib.patches import Patch

# Gradient boosting libraries
from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping as lgb_early_stop

# Feature attribution
import shap

# Classical models used later in the ensemble
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.preprocessing import StandardScaler

# Validation and metrics
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

---

## 2. Configuration

Every tunable value — paths, the fold count, the random seed, and each model's
hyperparameters — lives in one place. This keeps "magic numbers" out of the analysis cells
and makes the whole notebook reproducible from a single seed.

**A note on the regularization choices.** All three boosting models are configured
*conservatively on purpose*. Learning rates are low, depth is capped, and both L1 and L2
penalties are switched on. In tree models these two penalties do different jobs: the L1 term
(`reg_alpha` / `lambda_l1`) pushes leaf outputs toward zero and effectively simplifies the
trees, while the L2 term (`reg_lambda` / `lambda_l2`) discourages any single leaf from
producing an overconfident value. With a target this noisy, leaning on both is a deliberate
guard against fitting patterns that won't survive the regime change. (CatBoost exposes only
an L2 leaf penalty, so that's what it uses.)


In [ ]:
@dataclass
class CFG:
    seed: int = 42
    n_folds: int = 5

    # Update these paths to match your Kaggle input directory.
    train_path: str = "/kaggle/input/stock-prediction-dataset/train.csv"
    test_path: str = "/kaggle/input/stock-prediction-dataset/test.csv"
    sample_sub_path: str = "/kaggle/input/stock-prediction-dataset/sample_submission.csv"

    # Column roles
    target_col: str = "target"
    id_col: str = "id"
    stock_col: str = "stock_id"

    # --- Model hyperparameters -------------------------------------------------
    # Low learning rate + capped complexity + L1/L2 penalties = a conservative
    # configuration chosen to resist overfitting noisy, regime-shifted data.

    catboost_params: dict = field(default_factory=lambda: {
        "iterations": 2000,
        "learning_rate": 0.03,
        "depth": 6,
        "l2_leaf_reg": 5.0,          # CatBoost offers L2 leaf regularization only
        "eval_metric": "AUC",
        "random_seed": 42,
        "verbose": 200,
        "early_stopping_rounds": 100,
        "task_type": "GPU",          # set to "CPU" if no GPU is attached
    })

    xgboost_params: dict = field(default_factory=lambda: {
        "n_estimators": 2000,
        "learning_rate": 0.03,
        "max_depth": 6,
        "reg_alpha": 1.0,            # L1: simplifies trees
        "reg_lambda": 5.0,           # L2: tempers leaf confidence
        "eval_metric": "auc",
        "random_state": 42,
        "verbosity": 0,
        "early_stopping_rounds": 100,
        "tree_method": "hist",
        "device": "cuda",            # set to "cpu" if no GPU is attached
    })

    lightgbm_params: dict = field(default_factory=lambda: {
        "n_estimators": 2000,
        "learning_rate": 0.03,
        "num_leaves": 63,
        "lambda_l1": 1.0,            # L1
        "lambda_l2": 5.0,            # L2
        "metric": "auc",
        "random_state": 42,
        "verbose": -1,
        "n_jobs": -1,
    })

    # A lightweight model is all that's needed to *detect* distribution shift.
    adversarial_params: dict = field(default_factory=lambda: {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "metric": "auc",
        "random_state": 42,
        "verbose": -1,
    })


cfg = CFG()
print("Configuration loaded.")

---

## 3. Loading the Data (with a Memory Pass)

The training set is ~440K rows across 29 columns. Left as float64, the train and test frames
together sit around 100 MB — manageable alone, but wasteful once we start holding five
cross-validation folds and several models in memory at once. Two cheap conversions roughly
halve the footprint with no meaningful loss:

- **float64 → float32.** These features are bounded ratios and indicators, not prices that
  need many significant digits. float32 precision is more than enough.
- **`stock_id` string → uint8.** The identifiers run `stock_000` … `stock_099`, so the
  numeric part fits in a single byte.

The helper below reports the before/after memory and the class balance, which is worth
eyeballing at the start of any classification task.


In [ ]:
def optimize_memory(df: pd.DataFrame, stock_col: str = "stock_id") -> pd.DataFrame:
    # Downcast numeric dtypes to shrink memory footprint by ~50%.
    # "stock_030" -> 30, stored as a single-byte integer
    if stock_col in df.columns:
        df[stock_col] = df[stock_col].str.extract(r"(\d+)").astype(np.uint8)

    # float64 -> float32 (sufficient precision for ratio/indicator features)
    float_cols = df.select_dtypes(include="float64").columns
    df[float_cols] = df[float_cols].astype(np.float32)

    # shrink any remaining integer columns to their smallest safe type
    for col in df.select_dtypes(include="int64").columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")

    return df


train = pd.read_csv(cfg.train_path)
test = pd.read_csv(cfg.test_path)

mem_before = (train.memory_usage(deep=True).sum()
              + test.memory_usage(deep=True).sum()) / 1e6

train = optimize_memory(train)
test = optimize_memory(test)

mem_after = (train.memory_usage(deep=True).sum()
             + test.memory_usage(deep=True).sum()) / 1e6

print(f"Memory: {mem_before:.1f} MB -> {mem_after:.1f} MB "
      f"({(1 - mem_after / mem_before) * 100:.0f}% reduction)")
print(f"Train: {train.shape}   Test: {test.shape}")
print("\nClass balance (train):")
print(train[cfg.target_col].value_counts(normalize=True).round(4).to_string())

---

## 4. Adversarial Validation — How Different Are the Two Eras?

Before building a predictive model, it's worth measuring the problem we suspect exists. The
idea behind **adversarial validation** is simple and clever: forget the real target for a
moment, label every training row `0` and every test row `1`, and train a classifier to tell
them apart.

The AUC of *that* classifier is a direct measure of distribution shift:

- **~0.50** → train and test are indistinguishable. A model trained on one should transfer
  cleanly to the other. Nothing to worry about.
- **Well above 0.50** → the two periods look systematically different, and whichever features
  the classifier leans on are the ones carrying era-specific information. Those features are
  exactly where overfitting to the training regime will sneak in.

The feature importances from this model become a **shift-risk score** we carry forward. This
is a distinct question from "is the feature useful for predicting direction?" — a feature can
be highly shift-prone and useless, or stable and useful, or any other combination. We'll
reconcile the two views shortly.


In [ ]:
def adversarial_validation(train_df, test_df, cfg):
    # Train a classifier to separate train rows from test rows.
    #
    # Returns a feature-importance table (the shift-risk score) and the mean AUC.
    # A high AUC means the two periods are easy to tell apart -> large distribution shift.
    ignore = [cfg.id_col, cfg.target_col, cfg.stock_col]
    features = [c for c in train_df.columns if c not in ignore]

    # Relabel: 0 = came from train, 1 = came from test
    a_train = train_df[features].copy()
    a_test = test_df[features].copy()
    a_train["is_test"] = 0
    a_test["is_test"] = 1

    combined = pd.concat([a_train, a_test], ignore_index=True)
    X, y = combined[features], combined["is_test"]

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=cfg.seed)
    aucs, importances = [], np.zeros(len(features))

    for fold, (tr, val) in enumerate(skf.split(X, y)):
        model = LGBMClassifier(**cfg.adversarial_params)
        model.fit(X.iloc[tr], y.iloc[tr],
                  eval_set=[(X.iloc[val], y.iloc[val])],
                  callbacks=[lgb_early_stop(100)])
        preds = model.predict_proba(X.iloc[val])[:, 1]
        auc = roc_auc_score(y.iloc[val], preds)
        aucs.append(auc)
        importances += model.feature_importances_
        print(f"  Fold {fold + 1} AUC: {auc:.4f}")

    importances /= 3
    table = (pd.DataFrame({"feature": features, "importance": importances})
             .sort_values("importance", ascending=False)
             .reset_index(drop=True))

    mean_auc = float(np.mean(aucs))
    print(f"\nAdversarial AUC: {mean_auc:.4f}")
    print("=> Large distribution shift." if mean_auc > 0.55
          else "=> Distributions look similar.")
    return table, mean_auc


adv_importance, adv_auc = adversarial_validation(train, test, cfg)

### Reading the result

The adversarial AUC lands around **0.86** — nowhere near 0.50. In plain terms: a simple model
can look at the technical indicators for a single day and guess *which era it came from* with
high confidence. The 2000–2023 and 2024–2026 periods are genuinely, measurably different.

This is the empirical justification for everything that follows. It confirms that the real
enemy here is not model weakness but **overfitting to a regime that won't repeat**. The next
cell groups features into risk tiers based on how much they drive that separation.


In [ ]:
# Bucket features into shift-risk tiers based on adversarial importance.
# Thresholds are chosen from the natural gaps in the importance distribution.
high_risk = adv_importance.loc[adv_importance.importance > 1000, "feature"].tolist()
med_risk = adv_importance.loc[(adv_importance.importance > 400)
                              & (adv_importance.importance <= 1000), "feature"].tolist()
low_risk = adv_importance.loc[adv_importance.importance <= 400, "feature"].tolist()

tier_color = {"HIGH": "#d62728", "MEDIUM": "#ff7f0e", "LOW": "#2ca02c"}
bar_colors = [tier_color["HIGH"] if v > 1000
              else tier_color["MEDIUM"] if v > 400
              else tier_color["LOW"]
              for v in adv_importance.importance]

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=adv_importance, x="importance", y="feature", palette=bar_colors, ax=ax)
ax.axvline(1000, color="red", ls="--", alpha=0.5, label="High-risk threshold")
ax.axvline(400, color="orange", ls="--", alpha=0.5, label="Medium-risk threshold")
ax.set_title(f"Distribution-Shift Risk by Feature  (adversarial AUC = {adv_auc:.3f})")
ax.set_xlabel("Adversarial importance  (higher = more era-specific)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

print("HIGH shift-risk  :", high_risk)
print("MEDIUM shift-risk:", med_risk)
print("LOW shift-risk   :", low_risk)

The pattern is intuitive once you see it. The **high-risk** features are the long-horizon,
volatility- and trend-based ones — 60-day volatility, ATR, the 200-day moving-average ratio.
These naturally sit at different absolute levels across a 20-year span, so they leak era
information. The **low-risk** features are the short-horizon ones — one-day returns, short
rate-of-change — which behave similarly whether it's 2005 or 2025. A 1% down-day looks like a
1% down-day in any market.

We won't act on this yet. Instead we'll pair it with a *predictive*-importance ranking and let
the two views inform the feature set together.


---

## 5. Feature Selection via Three-Model SHAP Consensus

To decide which features genuinely help predict *direction*, we need an importance measure
that is honest about redundancy. Built-in tree importances (split counts or gain) tend to
over-credit correlated features, splitting the credit for one underlying signal across its
several near-duplicate columns. **SHAP values** attribute each prediction to its inputs in a
way that's far more robust to that problem — they estimate each feature's true marginal
contribution.

Two design choices make this ranking trustworthy rather than just plausible:

- **Three libraries, not one.** CatBoost, XGBoost, and LightGBM grow trees with different
  heuristics. A feature that ranks highly across all three is important because of the *data*,
  not because of one algorithm's quirks. We average the per-model ranks into a consensus.
- **SHAP on a validation subsample.** SHAP is expensive, and its *rankings* stabilize long
  before its values are pinned down to many decimals. Computing on 10,000 validation rows per
  fold gives the same ordering as the full set at a fraction of the runtime — which matters
  when it's 3 models × 5 folds.

Each model is also scored by AUC per fold, giving an early read on how much signal is
realistically available.


In [ ]:
ignore = [cfg.id_col, cfg.target_col, cfg.stock_col]
feature_cols = [c for c in train.columns if c not in ignore]
X = train[feature_cols]
y = train[cfg.target_col]

SHAP_SAMPLE = 10_000  # validation rows per fold used for SHAP (rankings are stable here)
skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)


def cv_shap(make_model, fit_fn, X, y, skf, sample):
    # Run stratified CV for one model. Returns (per-fold AUCs, mean |SHAP| per feature).
    # `fit_fn` isolates each library's slightly different fit signature.
    aucs = []
    shap_sum = np.zeros(X.shape[1])
    for fold, (tr, val) in enumerate(skf.split(X, y)):
        model = make_model()
        fit_fn(model, X.iloc[tr], y.iloc[tr], X.iloc[val], y.iloc[val])

        auc = roc_auc_score(y.iloc[val], model.predict_proba(X.iloc[val])[:, 1])
        aucs.append(auc)
        print(f"  Fold {fold + 1} AUC: {auc:.4f}")

        idx = val[:sample] if len(val) > sample else val
        explainer = shap.TreeExplainer(model)
        shap_sum += np.abs(explainer.shap_values(X.iloc[idx])).mean(axis=0)

    return aucs, shap_sum / skf.get_n_splits()


# --- CatBoost -----------------------------------------------------------------
print("CatBoost")
cb_aucs, cb_shap = cv_shap(
    lambda: CatBoostClassifier(**cfg.catboost_params),
    lambda m, Xt, yt, Xv, yv: m.fit(Pool(Xt, yt), eval_set=Pool(Xv, yv), verbose=0),
    X, y, skf, SHAP_SAMPLE,
)
print(f"  mean AUC: {np.mean(cb_aucs):.4f}\n")

# --- XGBoost ------------------------------------------------------------------
print("XGBoost")
xgb_aucs, xgb_shap = cv_shap(
    lambda: XGBClassifier(**cfg.xgboost_params),
    lambda m, Xt, yt, Xv, yv: m.fit(Xt, yt, eval_set=[(Xv, yv)], verbose=False),
    X, y, skf, SHAP_SAMPLE,
)
print(f"  mean AUC: {np.mean(xgb_aucs):.4f}\n")

# --- LightGBM -----------------------------------------------------------------
print("LightGBM")
lgb_aucs, lgb_shap = cv_shap(
    lambda: LGBMClassifier(**cfg.lightgbm_params),
    lambda m, Xt, yt, Xv, yv: m.fit(Xt, yt, eval_set=[(Xv, yv)],
                                     callbacks=[lgb_early_stop(100)]),
    X, y, skf, SHAP_SAMPLE,
)
print(f"  mean AUC: {np.mean(lgb_aucs):.4f}")

In [ ]:
# Combine the three SHAP profiles into a single consensus ranking.
ranking = pd.DataFrame({"feature": feature_cols})
for name, vals in [("CatBoost", cb_shap), ("XGBoost", xgb_shap), ("LightGBM", lgb_shap)]:
    ranking[f"{name}_shap"] = vals
    ranking[f"{name}_rank"] = ranking[f"{name}_shap"].rank(ascending=False).astype(int)

rank_cols = [c for c in ranking.columns if c.endswith("_rank")]
ranking["avg_rank"] = ranking[rank_cols].mean(axis=1)
ranking["avg_shap"] = ranking[["CatBoost_shap", "XGBoost_shap", "LightGBM_shap"]].mean(axis=1)

# Attach each feature's shift-risk tier from the adversarial step.
def risk_tier(f):
    return "HIGH" if f in high_risk else "MEDIUM" if f in med_risk else "LOW"

ranking["shift_risk"] = ranking["feature"].map(risk_tier)
ranking = ranking.sort_values("avg_rank").reset_index(drop=True)

print("Consensus feature ranking (1 = most predictive), with shift-risk tier:\n")
print(ranking[["feature", "CatBoost_rank", "XGBoost_rank", "LightGBM_rank",
               "avg_rank", "shift_risk"]].to_string(index=False))

### The two-signal view

Individual model AUCs cluster in the low-0.54s — a faint but real edge, consistent with the
"weak signal" expectation. More useful is the combined picture below. The table pairs each
feature's **predictive** rank (from SHAP) with its **shift-risk** tier (from adversarial
validation), and the scatter plot makes the trade-off visual.

The plot has four implicit quadrants:

| | **High SHAP** (predictive) | **Low SHAP** (not predictive) |
|---|---|---|
| **Low shift-risk** (stable) | ✅ ideal — keep | neutral — drop for simplicity |
| **High shift-risk** (era-specific) | ⚠️ tempting but dangerous | ❌ pure liability — drop |

The feature we most want is top-left: predictive *and* stable. `return_1d` is the clearest
example — the single most predictive feature by consensus, and low shift-risk. The dangerous
quadrant is top-right: features like 60-day volatility that a model will happily lean on in
cross-validation but which encode the very era information that won't transfer.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# (a) Consensus predictive ranking, colored by shift-risk.
r = ranking.sort_values("avg_rank")
axes[0].barh(r["feature"], r["avg_rank"].max() - r["avg_rank"] + 1,
             color=[tier_color[t] for t in r["shift_risk"]])
axes[0].invert_yaxis()
axes[0].set_title("Predictive Importance (SHAP consensus)")
axes[0].set_xlabel("Importance  (longer = more predictive)")
legend = [Patch(facecolor=tier_color[t], label=f"{t} shift-risk")
          for t in ["HIGH", "MEDIUM", "LOW"]]
axes[0].legend(handles=legend, loc="lower right")

# (b) The decision matrix: predictive power vs. shift-risk.
adv_lookup = dict(zip(adv_importance.feature, adv_importance.importance))
axes[1].scatter(ranking["feature"].map(adv_lookup), ranking["avg_shap"],
                c=[tier_color[t] for t in ranking["shift_risk"]],
                s=120, edgecolors="black", linewidths=0.5)
for _, row in ranking.iterrows():
    axes[1].annotate(row["feature"], (adv_lookup[row["feature"]], row["avg_shap"]),
                     fontsize=7, xytext=(4, 4), textcoords="offset points")
axes[1].axvline(600, color="gray", ls=":", alpha=0.5)
axes[1].axhline(ranking["avg_shap"].median(), color="gray", ls=":", alpha=0.5)
axes[1].set_title("Decision Matrix: Predictive Power vs. Shift-Risk")
axes[1].set_xlabel("Shift-risk  (adversarial importance)  ->")
axes[1].set_ylabel("Predictive power  (mean |SHAP|)  ->")
axes[1].legend(handles=legend, loc="upper right")

plt.tight_layout()
plt.show()

---

## 6. Collinearity Reduction

The decision matrix tells us about *usefulness* and *stability*; it says nothing about
*redundancy*. Several of these indicators are near-perfect substitutes for one another —
different arithmetic over the same underlying price series. Keeping both members of such a
pair adds no information but doubles the model's exposure to that one signal.

The rule here is deliberately mechanical: compute the absolute correlation matrix, find every
pair above **r = 0.95**, and from each pair drop the member with the *weaker* SHAP consensus
rank. Letting the predictive ranking break the tie means we keep the more useful of two
interchangeable features.

Alongside the collinearity drops we remove two other small groups, for reasons already
established:

- **Bottom-of-the-table features** all three models agree contribute almost nothing.
- **`volatility_20d`**, which is the worst of both worlds — low predictive value *and* high
  shift-risk.


In [ ]:
corr = train[feature_cols].corr().abs()
CORR_THRESHOLD = 0.95

# Upper triangle only, so each pair is considered once.
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = [(a, b, corr.loc[b, a])
         for a in upper.columns for b in upper.index
         if upper.loc[b, a] > CORR_THRESHOLD]

rank_of = dict(zip(ranking.feature, ranking.avg_rank))
print(f"Correlated pairs above r = {CORR_THRESHOLD} (dropping the weaker-ranked feature):\n")
collinear_drops = set()
for a, b, r in sorted(pairs, key=lambda x: -x[2]):
    drop = a if rank_of[a] > rank_of[b] else b
    collinear_drops.add(drop)
    print(f"  {a} (rank {rank_of[a]:.0f})  vs  {b} (rank {rank_of[b]:.0f})   "
          f"r = {r:.3f}   -> drop {drop}")

# Correlation heatmap for context.
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)),
            cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, cbar_kws={"label": "|correlation|"}, ax=ax)
ax.set_title(f"Feature Correlation  (pairs above {CORR_THRESHOLD} pruned)")
plt.tight_layout()
plt.show()

In [ ]:
low_value_drops = {"roc_10", "return_10d", "sma_ratio_20", "roc_5"}  # bottom of SHAP table
worst_of_both = {"volatility_20d"}                                   # low SHAP + high shift-risk

all_drops = collinear_drops | low_value_drops | worst_of_both
selected_features = [f for f in feature_cols if f not in all_drops]

print(f"Collinearity drops : {sorted(collinear_drops)}")
print(f"Low-value drops    : {sorted(low_value_drops)}")
print(f"Worst-of-both drops: {sorted(worst_of_both)}")
print(f"\nDropped {len(all_drops)} features; kept {len(selected_features)}.")
print(f"\nRetained feature set:\n{selected_features}")

---

## 7. Per-Stock Normalization

One indicator value can mean very different things for different stocks. A 2% daily move is a
major event for a sleepy utility that usually drifts 0.2% a day, but an unremarkable Tuesday
for a volatile growth name that routinely swings 3%. If the model sees the raw `0.02` in both
cases, it has to learn stock-by-stock context on its own — a hard ask given the noise.

**Per-stock z-scoring** solves this by expressing each feature as a number of standard
deviations from *that stock's own* historical mean. After the transform, "+2.5" means "unusually
high for this particular stock" regardless of which stock it is, and the model can learn one
clean rule instead of a hundred bespoke ones. This is the same idea as the cross-sectional
normalization used throughout systematic trading.

**Avoiding leakage is critical.** The per-stock mean and standard deviation are computed on the
**training data only** and then applied to both train and test. Fitting these statistics on the
test set would let information about the 2024–2026 period bleed into the features — a subtle but
serious form of leakage. The function below enforces the fit-on-train, apply-to-both discipline
explicitly.


In [ ]:
def per_stock_zscore(train_df, test_df, features, stock_col):
    # Z-score each feature within each stock.
    #
    # Statistics are learned on TRAIN ONLY and applied to both frames, so no test-period
    # information leaks into the transform. Zero-variance cases map to 0 rather than dividing
    # by zero.
    tr, te = train_df.copy(), test_df.copy()
    stats = train_df.groupby(stock_col)[features].agg(["mean", "std"])

    for f in features:
        mean_map, std_map = stats[(f, "mean")], stats[(f, "std")]

        m, s = tr[stock_col].map(mean_map), tr[stock_col].map(std_map)
        tr[f] = np.where(s > 0, (tr[f] - m) / s, 0)

        m, s = te[stock_col].map(mean_map), te[stock_col].map(std_map)  # train stats on test
        te[f] = np.where(s > 0, (te[f] - m) / s, 0)

    return tr, te


train_z, test_z = per_stock_zscore(train, test, selected_features, cfg.stock_col)

# Sanity check: within a single stock, each feature should now be ~mean 0, std 1 on train.
check = train_z[train_z[cfg.stock_col] == 0][selected_features[:5]]
print("Post-normalization check for stock_000 (expect mean ~0, std ~1):\n")
print(check.describe().loc[["mean", "std"]].round(3).to_string())

---

## 8. Ablation Study — Which Choices Actually Help?

Two hypotheses now need testing rather than assuming: that per-stock normalization helps, and
that dropping the high shift-risk features improves *generalization* even if it costs
cross-validation score. An ablation isolates each factor by holding everything else constant.

Three configurations, one fast LightGBM each:

- **B — Z-scored, all retained features.** Normalization on, shift-prone features kept.
- **C — Raw, all retained features.** Normalization off. Isolates the value of z-scoring
  (B vs. C).
- **D — Z-scored, high shift-risk features removed.** Normalization on, the era-specific
  features dropped. Isolates the effect of shedding shift-risk (B vs. D).

The number to watch is not just the cross-validation AUC printed here, but how it will later
compare to held-out performance. That comparison is the whole point.


In [ ]:
high_shift_features = ["volatility_60d", "atr_14", "sma_ratio_200", "avg_range_10d"]
safe_features = [f for f in selected_features if f not in high_shift_features]

# Config D needs its own z-scored frames (a smaller feature set).
train_d, test_d = per_stock_zscore(train, test, safe_features, cfg.stock_col)

configs = {
    "B: z-scored, all features": (train_z[selected_features].values.astype(np.float32),),
    "C: raw, all features":      (train[selected_features].values.astype(np.float32),),
    "D: z-scored, drop shift":   (train_d[safe_features].values.astype(np.float32),),
}

y_arr = train[cfg.target_col].values.astype(np.int8)
skf_ab = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)

ablation = {}
for name, (X_cfg,) in configs.items():
    fold_aucs = []
    for tr, val in skf_ab.split(X_cfg, y_arr):
        m = LGBMClassifier(**cfg.lightgbm_params)
        m.fit(X_cfg[tr], y_arr[tr], eval_set=[(X_cfg[val], y_arr[val])],
              callbacks=[lgb_early_stop(100)])
        fold_aucs.append(roc_auc_score(y_arr[val], m.predict_proba(X_cfg[val])[:, 1]))
    ablation[name] = (np.mean(fold_aucs), np.std(fold_aucs))
    print(f"{name:32s}  CV AUC {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}  "
          f"({X_cfg.shape[1]} features)")

### The key finding

Cross-validation ranks the configurations **B ≈ C(+z) > D**. Z-scoring clearly helps (B beats
C). And dropping the shift-prone features *lowers* CV score (D is worst) — which, taken at face
value, would argue for keeping them.

But held-out leaderboard performance tells the opposite story. Submitting these same
configurations to the competition's 2024–2026 test set produced:

| Config | CV AUC | Held-out AUC |
|---|---|---|
| B — z-scored, all features | ~0.5436 | 0.510 |
| D — z-scored, drop shift-risk | ~0.5367 | **0.513** |

**The configuration with the *worst* cross-validation score generalized the *best*.** This is
the single most important result in the notebook, and it's a direct consequence of the regime
shift measured back in step 4. Cross-validation rewards fitting the training era; the
high shift-risk features let the model do exactly that, which flatters CV and hurts the future.
Removing them sacrifices in-sample score to buy out-of-sample robustness.

The lesson generalizes well beyond this dataset: **when train and test come from different
regimes, cross-validation is an optimistic and partially misleading guide.** From here on, the
feature set is Config D — z-scored, shift-prone features removed — and decisions favor
generalization over CV.


---

## 9. Preparing the Final Matrices

Lock in the Config D feature set and build clean NumPy arrays for modeling, with explicit
guards against nulls and infinities. Both can arise from the z-score step (e.g. a stock with
zero variance in some feature) and both will silently corrupt training if left unchecked.


In [ ]:
FINAL_FEATURES = safe_features

X = train_d[FINAL_FEATURES].values.astype(np.float32)
y = train_d[cfg.target_col].values.astype(np.int8)
X_test = test_d[FINAL_FEATURES].values.astype(np.float32)

# Replace any NaN/inf produced upstream; report so nothing is silent.
for name, arr in [("train", X), ("test", X_test)]:
    n_null = int(np.isnan(arr).sum())
    n_inf = int(np.isinf(arr).sum())
    if n_null or n_inf:
        print(f"{name}: cleaned {n_null} NaN and {n_inf} inf values")
    else:
        print(f"{name}: no NaN or inf values")

X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

print(f"\nFinal feature count : {len(FINAL_FEATURES)}")
print(f"X            : {X.shape}")
print(f"X_test       : {X_test.shape}")
print(f"positive rate: {y.mean():.4f}")

---

## 10. Modeling — Diversity Over Raw Power

An early experiment (not shown) blended CatBoost, XGBoost, and LightGBM together and *lost* to
a single LightGBM. The reason is instructive: those three are all gradient-boosted trees. They
carve up the feature space in similar ways and, crucially, they make **similar mistakes**.
Averaging models that err in the same direction buys almost nothing.

Useful ensembling requires models that are wrong *differently*. So the final ensemble
deliberately spans four different model families:

- **LightGBM** — gradient-boosted trees. The strongest single learner; captures non-linear
  interactions but is the most prone to overfitting the training regime.
- **Logistic Regression (L1)** — a linear model. Sees only linear structure, but that structure
  is stable and it cannot chase the noisy interactions the trees might.
- **Gaussian Naive Bayes** — assumes features are independent. This is "wrong" as an assumption,
  but the payoff is a model that *structurally cannot* overfit to the joint noise patterns a
  tree will. It contributes precisely because it is so different.
- **Multi-Layer Perceptron** — a small neural network. Learns smooth non-linear mappings that
  are different in character from a tree's axis-aligned splits.

Everything runs inside a single stratified 5-fold loop, producing out-of-fold (OOF) predictions
for honest evaluation and averaged test-fold predictions for submission. Stratified folds keep
the class balance identical across folds, which stabilizes the AUC estimates.


In [ ]:
skf_final = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
model_names = ["LightGBM", "LogReg", "NaiveBayes", "MLP"]

oof = {m: np.zeros(len(X)) for m in model_names}       # out-of-fold predictions
test_pred = {m: np.zeros(len(X_test)) for m in model_names}

for fold, (tr, val) in enumerate(skf_final.split(X, y)):
    X_tr, X_val, y_tr, y_val = X[tr], X[val], y[tr], y[val]

    # Linear and neural models need standardized inputs; fit the scaler on the
    # training fold only to avoid leaking validation statistics.
    scaler = StandardScaler().fit(X_tr)
    X_tr_s, X_val_s, X_test_s = scaler.transform(X_tr), scaler.transform(X_val), scaler.transform(X_test)

    # LightGBM (raw features are fine for trees)
    lgb = LGBMClassifier(**cfg.lightgbm_params)
    lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb_early_stop(100)])
    oof["LightGBM"][val] = lgb.predict_proba(X_val)[:, 1]
    test_pred["LightGBM"] += lgb.predict_proba(X_test)[:, 1] / cfg.n_folds

    # Logistic Regression with L1 penalty (sparse, linear)
    lr = LogisticRegression(penalty="l1", C=0.1, solver="saga",
                            max_iter=1000, random_state=cfg.seed)
    lr.fit(X_tr_s, y_tr)
    oof["LogReg"][val] = lr.predict_proba(X_val_s)[:, 1]
    test_pred["LogReg"] += lr.predict_proba(X_test_s)[:, 1] / cfg.n_folds

    # Gaussian Naive Bayes (independent-feature assumption -> cannot overfit joint noise)
    nb = GaussianNB()
    nb.fit(X_tr, y_tr)
    oof["NaiveBayes"][val] = nb.predict_proba(X_val)[:, 1]
    test_pred["NaiveBayes"] += nb.predict_proba(X_test)[:, 1] / cfg.n_folds

    # Small MLP with strong L2 (alpha) regularization
    mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", alpha=1.0,
                        learning_rate="adaptive", max_iter=200, early_stopping=True,
                        validation_fraction=0.1, random_state=cfg.seed)
    mlp.fit(X_tr_s, y_tr)
    oof["MLP"][val] = mlp.predict_proba(X_val_s)[:, 1]
    test_pred["MLP"] += mlp.predict_proba(X_test_s)[:, 1] / cfg.n_folds

    print(f"Fold {fold + 1}  "
          + "  ".join(f"{m} {roc_auc_score(y_val, oof[m][val]):.4f}" for m in model_names))

print("\nOut-of-fold AUC by model:")
model_auc = {m: roc_auc_score(y, oof[m]) for m in model_names}
for m in model_names:
    print(f"  {m:12s}: {model_auc[m]:.5f}")

### Combining the models

With four sets of out-of-fold predictions in hand, we compare two ways of combining them, then
sanity-check both against the simple blend that had already proven itself on the leaderboard.

- **Dynamic weighting.** Weight each model by how far its OOF AUC clears the 0.500 baseline, so
  a model with a 0.040 edge counts twice as much as one with 0.020. This rewards signal without
  hand-tuning.
- **Ridge stacking.** Fit a ridge regression on the OOF predictions to learn an optimal linear
  combination. More flexible, but with more freedom to overfit the validation set — exactly the
  failure mode this whole project is wary of.

We report all three so the trade-offs are visible rather than asserted.


In [ ]:
# --- Dynamic weights: proportional to each model's edge over 0.5 -----------------
excess = {m: max(model_auc[m] - 0.5, 0) for m in model_names}
total = sum(excess.values())
weights = {m: excess[m] / total for m in model_names}

print("Dynamic blend weights:")
for m in model_names:
    print(f"  {m:12s}: AUC {model_auc[m]:.5f} -> weight {weights[m]:.3f}")

oof_dynamic = sum(weights[m] * oof[m] for m in model_names)
test_dynamic = sum(weights[m] * test_pred[m] for m in model_names)
auc_dynamic = roc_auc_score(y, oof_dynamic)

# --- Ridge stacking -------------------------------------------------------------
OOF = np.column_stack([oof[m] for m in model_names])
TEST = np.column_stack([test_pred[m] for m in model_names])
ridge = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0]).fit(OOF, y)
oof_ridge = ridge.predict(OOF)
test_ridge = ridge.predict(TEST)
auc_ridge = roc_auc_score(y, oof_ridge)

# --- Proven two-model blend: LightGBM + Naive Bayes -----------------------------
# A simple weight sweep; the tree provides the signal, NB regularizes it toward robustness.
best_w, best_auc = 0.0, 0.0
for w in np.arange(0.10, 0.50, 0.05):
    blend = w * oof["NaiveBayes"] + (1 - w) * oof["LightGBM"]
    a = roc_auc_score(y, blend)
    if a > best_auc:
        best_auc, best_w = a, w
test_nb_lgb = best_w * test_pred["NaiveBayes"] + (1 - best_w) * test_pred["LightGBM"]

print("\nEnsemble comparison (OOF AUC):")
print(f"  Dynamic blend (4 models) : {auc_dynamic:.5f}")
print(f"  Ridge stack   (4 models) : {auc_ridge:.5f}")
print(f"  NB + LGB blend           : {best_auc:.5f}   "
      f"(NB {best_w:.0%} / LGB {1 - best_w:.0%})")

---

## 11. Generating Submissions

Because cross-validation and held-out performance diverge on this data, the honest move is to
prepare all three candidate submissions and let the leaderboard adjudicate, rather than trusting
the best OOF score blindly. Each file is clipped to a valid probability range and checked for
the correct row count before writing.


In [ ]:
def write_submission(preds, path):
    sub = pd.read_csv(cfg.sample_sub_path)
    sub[cfg.target_col] = np.clip(preds, 0, 1)
    assert len(sub) == 53_276, f"expected 53,276 rows, got {len(sub)}"
    assert sub[cfg.target_col].notna().all(), "submission contains NaN"
    sub.to_csv(path, index=False)
    return sub[cfg.target_col]


candidates = {
    "submission_dynamic_blend.csv": test_dynamic,
    "submission_ridge_stack.csv":   test_ridge,
    "submission_nb_lgb_blend.csv":  test_nb_lgb,
}

print("Wrote submissions:\n")
for path, preds in candidates.items():
    p = write_submission(preds, path)
    print(f"  {path:32s}  mean {p.mean():.4f}  std {p.std():.4f}  "
          f"[{p.min():.3f}, {p.max():.3f}]")

---

## Results & Reflections

The progression of leaderboard scores tells the story better than any single number:

| Approach | Held-out AUC | What it taught |
|---|---:|---|
| Three-tree ensemble, all features | 0.511 | Similar models make similar errors — no diversity benefit |
| Config B (all features, z-scored) | 0.510 | Shift-prone features flatter CV but hurt the future |
| Config D (shift-risk features dropped) | 0.513 | Lower CV can mean *better* generalization |
| LightGBM + Naive Bayes blend | **0.515** | A deliberately different model regularizes a strong one |

For a task where 0.500 is a coin flip and 0.520 is considered a meaningful real-world edge,
landing at 0.515 on a genuinely out-of-sample, regime-shifted test set is a solid result. But
the final score is less interesting than the pattern behind it.

**Every gain came from making the system simpler or more robust — never more complex.** Dropping
features helped. Adding a naive, structurally limited model helped. The most powerful individual
learner, left to its own devices, overfit the past. On noisy, non-stationary financial data,
that direction of travel is the point, not an accident.

**Themes worth carrying forward:**

1. **Measure the shift before modeling it.** Adversarial validation turned a vague worry about
   "regime change" into a concrete AUC and a ranked list of the features responsible, which then
   guided every downstream choice.
2. **Separate *predictive* from *stable*.** SHAP answers "does this feature help?"; adversarial
   validation answers "will it still help later?" Neither question substitutes for the other,
   and the best features score well on both.
3. **Distrust cross-validation under distribution shift.** The clearest lesson here is a
   configuration that was *worst* in CV and *best* out of sample. When the test set is a
   different world, CV is optimistic by construction.
4. **Ensemble for difference, not for quantity.** Three trees added little; one tree plus one
   naive model added more. Error diversity is what makes a blend work.

**Where this could go next:** a proper stacking layer trained with leak-free nested
cross-validation, explicit modeling of the per-stock base rates, or a time-decay weighting that
leans on more recent training data if any temporal structure survives the row shuffling. Each is
a natural extension of the same core discipline — optimize for the world the model will actually
face, not the one it was trained on.
